# Lesson 6: AI in Biotechnology

**Module 1:** Introduction to Artificial Intelligence

**Lesson Objectives:**
- Describe AI applications in drug discovery, genomics, protein folding, medical imaging, and personalized medicine
- Build a virtual screening pipeline
- Understand AI's impact on biotechnology

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.datasets import make_classification

print("Libraries loaded successfully")

## Virtual Screening Simulation

Let us simulate a virtual screening pipeline. We generate synthetic molecular descriptors for compounds and predict which ones are active against a target.

In [ ]:
# Generate synthetic molecular dataset
# Features: 20 molecular descriptors (e.g., molecular weight, logP, H-bond donors, etc.)
np.random.seed(42)
n_compounds = 1000
n_features = 20

X, y = make_classification(
    n_samples=n_compounds,
    n_features=n_features,
    n_informative=10,
    n_redundant=3,
    n_clusters_per_class=1,
    weights=[0.95, 0.05],  # 5% active
    random_state=42
)

feature_names = [f'desc_{i}' for i in range(n_features)]
df = pd.DataFrame(X, columns=feature_names)
df['active'] = y

print(f"Dataset: {len(df)} compounds")
print(f"Active: {df['active'].sum()} ({df['active'].mean():.1%})")
print(f"Inactive: {(df['active'] == 0).sum()}\n")
print(df[feature_names[:5] + ['active']].head())

In [ ]:
# Split into training (known actives) and screening set (unknown)
X_train, X_screen, y_train, y_screen = train_test_split(
    X, y, test_size=0.8, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} compounds ({y_train.sum()} active)")
print(f"Screening set: {X_screen.shape[0]} compounds ({y_screen.sum()} active)")

In [ ]:
# Train a Random Forest classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict on screening set
y_prob = model.predict_proba(X_screen)[:, 1]
y_pred = model.predict(X_screen)

print("Model Performance on Training Set:")
print(classification_report(y_screen, y_pred))

# Confusion matrix
cm = confusion_matrix(y_screen, y_pred)
print("\nConfusion Matrix:")
print(pd.DataFrame(cm, index=['Inactive', 'Active'], columns=['Pred Inactive', 'Pred Active']))

In [ ]:
# Create screening results
screening_results = pd.DataFrame({
    'compound_id': np.arange(X_screen.shape[0]),
    'activity_probability': y_prob,
    'predicted_active': y_pred,
    'true_active': y_screen
})

# Sort by predicted activity probability (highest first)
screening_results = screening_results.sort_values(
    'activity_probability', ascending=False
).reset_index(drop=True)
screening_results['rank'] = range(1, len(screening_results) + 1)

print("Top 20 Candidates for Experimental Validation:")
print(screening_results.head(20))

# Evaluate top-N hit rate
top_n_hits = []
for n in [5, 10, 20, 50, 100]:
    hits = screening_results.head(n)['true_active'].sum()
    top_n_hits.append((n, hits))

print("\nTop-N Hit Rates:")
for n, hits in top_n_hits:
    print(f"  Top {n:4d}: {hits} hits ({hits/n:.0%})")

In [ ]:
# Feature importance analysis
importance_df = pd.DataFrame({
    'descriptor': feature_names,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Most Important Molecular Descriptors:")
print(importance_df.head(10))

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(importance_df['descriptor'].head(10)[::-1], 
         importance_df['importance'].head(10)[::-1])
plt.xlabel('Feature Importance')
plt.title('Top 10 Molecular Descriptors for Activity Prediction')
plt.tight_layout()
plt.show()

In [ ]:
# Plot screening results distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(screening_results['activity_probability'], bins=30, edgecolor='black')
plt.axvline(0.5, color='red', linestyle='--', label='Threshold (0.5)')
plt.xlabel('Predicted Activity Probability')
plt.ylabel('Number of Compounds')
plt.title('Distribution of Predicted Activities')
plt.legend()

plt.subplot(1, 2, 2)
cumulative_hits = screening_results['true_active'].cumsum()
plt.plot(range(1, len(cumulative_hits) + 1), cumulative_hits)
plt.xlabel('Number of Compounds Screened')
plt.ylabel('Cumulative Hits Found')
plt.title('Screening Cumulative Hit Curve')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Interpreting the Virtual Screening Results

The virtual screening pipeline ranked all compounds by predicted probability of being active. Key insights:

1. **Enrichment**: The top 10% of compounds contain X% of all active compounds (much better than random)
2. **Hit rate**: In the top 20 candidates, Y% are actually active (vs. 5% baseline)
3. **Feature importance**: We identified which molecular descriptors are most predictive
4. **Validation**: These top-ranked compounds would be prioritized for experimental testing

In a real setting, this pipeline would screen millions of compounds and reduce the number of physical experiments by 100-1000x.

## Medical Imaging Concept

Let us demonstrate a simple image classifier for medical diagnosis.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# Using digits as a proxy for medical image classification
digits = load_digits()
Xd, yd = digits.data, digits.target

# Binary: is this digit a '0' or not? (like detecting presence of abnormality)
yd_binary = (yd == 0).astype(int)

Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    Xd, yd_binary, test_size=0.3, random_state=42
)

gb_model = GradientBoostingClassifier(n_estimators=50, random_state=42)
gb_model.fit(Xd_train, yd_train)
yd_pred = gb_model.predict(Xd_test)

print("Medical Image Classification (Normal vs. Abnormal):")
print(classification_report(yd_test, yd_pred, 
      target_names=['Normal', 'Abnormal']))
print(f"Accuracy: {accuracy_score(yd_test, yd_pred):.2%}")

## Exercises

1. Modify the virtual screening pipeline to use a different classifier (e.g., Gradient Boosting) and compare performance.
2. Add a threshold optimization step: find the probability threshold that maximizes F1 score.
3. Research AlphaFold's architecture and write a markdown cell explaining how it predicts protein structures.

In [ ]:
# Exercise 1: Compare classifiers
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC

classifiers = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, random_state=42),
}

for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_screen)
    acc = accuracy_score(y_screen, y_pred)
    print(f"{name:20s}: Accuracy = {acc:.2%}")

## Summary

- AI transforms biotechnology: drug discovery, genomics, protein folding, medical imaging, personalized medicine
- Virtual screening with ML can reduce experimental workload by 100-1000x
- Feature importance helps interpret which molecular properties drive activity
- AI in biotech requires domain expertise and experimental validation
- The field is rapidly advancing and will transform healthcare